# 提取背鳍特征



In [16]:
import os
import glob
import timm
import numpy as np
import pandas as pd
import torchvision.transforms as T

from wildlife_tools.features import DeepFeatures
from wildlife_tools.data import ImageDataset

In [6]:
root_dir = r'/media/filming/2025-白海豚/20240825-JM_02-3/'
metainfo = pd.read_csv(root_dir + "METAINFO/FIN_METAINFO.csv")

In [7]:
metainfo

,identity,path,crop_conf,x_min,x_max,y_min,y_max,orig_img,orig_img_h,orig_img_w,clearness,select
0,0,FIN/5276_20240825JM02ZRA16152_FIN00.JPG,0.795169,1781,1947,1523,1623,5276_20240825JM02ZRA16152.JPG,2880,4320,0.000047,False
1,1,FIN/5277_20240825JM02ZRA16153_FIN00.JPG,0.774381,1769,1924,1461,1557,5277_20240825JM02ZRA16153.JPG,2880,4320,0.003009,False
2,2,FIN/5280_20240825JM02ZRA16156_FIN00.JPG,0.820775,2128,2491,1019,1234,5280_20240825JM02ZRA16156.JPG,2880,4320,0.007198,False
3,3,FIN/5280_20240825JM02ZRA16156_FIN01.JPG,0.819164,2327,2655,1292,1515,5280_20240825JM02ZRA16156.JPG,2880,4320,0.266308,True
4,4,FIN/5281_20240825JM02ZRA16157_FIN00.JPG,0.852608,2483,2831,1555,1796,5281_20240825JM02ZRA16157.JPG,2880,4320,0.764504,True
...,...,...,...,...,...,...,...,...,...,...,...,...
3759,3759,FIN/8000_20240825JM02ZRA18876_FIN00.JPG,0.819379,1810,2135,1418,1610,8000_20240825JM02ZRA18876.JPG,2880,4320,0.402914,True
3760,3760,FIN/8001_20240825JM02ZRA18877_FIN00.JPG,0.823672,1883,2202,1408,1606,8001_20240825JM02ZRA18877.JPG,2880,4320,0.002417,False
3761,3761,FIN/8002_20240825JM02ZRA18878_FIN00.JPG,0.810614,1891,2201,1452,1639,8002_20240825JM02ZRA18878.JPG,2880,4320,0.920113,True
3762,3762,FIN/8003_20240825JM02ZRA18879_FIN00.JPG,0.837352,1949,2244,1355,1539,8003_20240825JM02ZRA18879.JPG,2880,4320,0.875908,True


In [4]:
model = timm.create_model('hf-hub:BVRA/MegaDescriptor-L-384', pretrained=True)

In [9]:
transform = T.Compose([
    T.Resize([384, 384]),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
])
dataset = ImageDataset(
        root = root_dir,
        metadata=metainfo.query('select==True'),
        transform=transform,
        col_label = 'identity',
        col_path = 'path'
)

In [10]:
extractor = DeepFeatures(model, device='cuda', batch_size=32) 
# batch size 32 to match 12G gpu memory
features = extractor(dataset)

100%|███████████████████████████████████████████████████████████████| 91/91 [02:04<00:00,  1.36s/it]
